<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_1_SLR/17_1_3_SLR_LINE_Assumptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Linear Regression: The LINE Assumptions (Diagnosing the Model)

Author: Brad Sheese

---

## What This Notebook Is About

In Notebooks 17_1_1 and 17_1_2 the Palmer Penguins data cooperated beautifully. We fit a line, the p-values were tiny, and we declared victory.

But here's the uncomfortable truth:

> **OLS will draw a line through anything. It doesn't care whether a line is a good idea.**

Hand it a curve, pure noise, or 300 points with one huge outlier — you still get a slope, $R^2$, p-value, and confidence interval. The summary table never complains.

The whole inferential machinery from 17_1_2 is built on a set of assumptions about how the data behave. When those assumptions hold, the numbers are trustworthy. When they don't, the numbers can be completely wrong *while still looking beautiful on the page*.

Statisticians summarize these assumptions with a handy mnemonic: **LINE**.

| Letter | Assumption | Plain English |
|:-:|---|---|
| **L** | **Linearity** | The true relationship between $x$ and $y$ is actually a line. |
| **I** | **Independence** | Each observation's error is independent of every other. |
| **N** | **Normality of residuals** | Residuals are bell-curve distributed. |
| **E** | **Equal variance** (homoscedasticity) | The spread of residuals is constant across all values of $x$. |

We'll use the **Auto MPG** dataset, which violates multiple assumptions at once.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

sns.set_style('whitegrid')
rng = np.random.default_rng(seed=42)

mpg = sns.load_dataset('mpg').dropna().reset_index(drop=True)
print(f'Cars: {len(mpg)}')
mpg[['mpg', 'displacement', 'horsepower', 'weight']].head()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(mpg['displacement'], mpg['mpg'], s=20, alpha=0.5, color='steelblue')
ax.set_xlabel('Engine displacement (cu. in.)')
ax.set_ylabel('Fuel efficiency (mpg)')
ax.set_title('Auto MPG: displacement vs. mpg')
plt.show()

Even before fitting, three things stand out:

1. Bigger engines mean lower mpg — a clear negative trend.
2. But the trend **curves**: past ~200 cu. in., mpg stops falling as fast.
3. The **spread** narrows at high displacement: small-engine cars range from ~20 to 45 mpg, big-engine cars cluster between 10 and 15.

Watch OLS ignore both problems, then watch the diagnostics catch them.

In [ ]:
X_sm = sm.add_constant(mpg['displacement'])
y = mpg['mpg']
model = sm.OLS(y, X_sm).fit()
print(model.summary())

If you stopped here, you'd report:

- $R^2 = 0.648$ — displacement explains 65% of mpg variation.
- Slope: $-0.060$ — 100 extra cu. in. costs ~6 mpg.
- $p < 0.001$ — wildly significant.
- CI: $[-0.064, -0.056]$ — very tight.

All of these are built on assumptions we haven't checked. Let's draw the fitted line.

In [ ]:
x_grid = np.linspace(mpg['displacement'].min(), mpg['displacement'].max(), 200)
y_pred = model.params['const'] + model.params['displacement'] * x_grid

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(mpg['displacement'], mpg['mpg'], s=20, alpha=0.4, color='steelblue', label='Cars')
ax.plot(x_grid, y_pred, color='darkorange', linewidth=2.5, label='OLS fit')
ax.set_xlabel('Engine displacement (cu. in.)')
ax.set_ylabel('Fuel efficiency (mpg)')
ax.set_title('OLS will draw a straight line through a curve')
ax.legend()
plt.show()

The line **under-predicts at both ends** and **over-predicts in the middle** — the classic signature of a line forced through a curve. The summary table told us everything was fine. That gap is what the LINE diagnostics are designed to expose.

---

## Section 1 — L: Linearity

> **The assumption:** the true relationship is a straight line.

The diagnostic: a **residual plot** ($e_i = y_i - \hat{y}_i$ vs. fitted values), with a smooth lowess curve to highlight patterns.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.residplot(x=mpg['displacement'], y=mpg['mpg'], lowess=True,
    scatter_kws={'s': 20, 'alpha': 0.4, 'color': 'steelblue'},
    line_kws={'color': 'red', 'lw': 2.5}, ax=ax)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Engine displacement (cu. in.)')
ax.set_ylabel('Residual (mpg)')
ax.set_title('Residual plot: the L-assumption in one picture')
plt.show()

The red lowess curve carves a distinct **U-shape**: residuals positive on the left, negative in the middle, positive on the right. This is exactly what happens when you force a line through a curve.

> **The rule:** a residual plot should be a *horizontal band of noise* — random scatter around zero with no trend. Anything else means the model has missed something systematic.

We'll learn how to fix this in Notebook 17_1_5 — most commonly by transforming $x$ or $y$.

---

## Section 2 — I: Independence

> **The assumption:** each observation's residual is independent of every other.

This is the hardest assumption to check by eye. It fails most often in **time-series data** (consecutive errors are correlated) or **clustered data** (students in the same classroom share unmeasured factors).

Our car dataset is cross-sectional — each car is independent — so this assumption should hold. But here's how you'd check.

### The Durbin–Watson statistic

Scroll up to the summary table and look at `Durbin-Watson: 0.926`. This tests for serial correlation in residuals (ordered as the data arrives).

Rough guidelines (for a single predictor, $n=392$):

- **DW $\approx 2.0$** — residuals look independent. Good.
- **DW below roughly 1.7** — positive autocorrelation (consecutive errors drift together). Warning.
- **DW above roughly 2.3** — negative autocorrelation (they bounce). Also a warning.
- **Between 1.7 and 2.3** — the "zone of indecision." The test is inconclusive.

Exact critical values depend on sample size and number of predictors; these are approximations.

Our DW of 0.926 looks alarming — but it almost certainly reflects an *artifact of row ordering*: the dataset is sorted by model year, so cars from similar eras sit next to each other. That's not a real independence problem. The correct response: *don't worry about DW unless your data has a natural temporal or spatial order*.

For cross-sectional data like one-row-per-penguin or one-row-per-car, independence is usually a safe assumption. The Durbin-Watson test matters most for time-series data, which you'll encounter in later courses.

---

## Section 3 — N: Normality of Residuals

> **The assumption:** residuals are drawn from a normal distribution centered on zero.

If residuals are strongly non-normal, the standard errors, p-values, and CIs from the summary table can be off. But with a sample size above ~30, the **Central Limit Theorem** makes inference fairly robust to modest non-normality.

We'll check using a **Q–Q plot** (quantile–quantile plot), which compares your sorted residuals against what you'd expect from a normal distribution. A Q–Q plot is more informative than a histogram because it shows deviations across the *entire* distribution — not just the middle.

### Reading a Q-Q plot in 30 seconds

- **Points on the 45° reference line:** residuals are perfectly normal.
- **Points curving off at the ends:** tails are heavier or lighter than normal.
- **S-shape or banana:** residuals are skewed.

In [ ]:
residuals = model.resid

fig, ax = plt.subplots(figsize=(7.5, 7.5))
sm.qqplot(residuals, line='s', ax=ax,
          markerfacecolor='steelblue', markeredgecolor='steelblue', alpha=0.5)
# Color the reference line (find it by linestyle — safer than relying on index order).
for line in ax.get_lines():
    if line.get_linestyle() != 'None':
        line.set_color('red')
        line.set_linewidth(2)
ax.set_title('Q-Q plot of residuals')
plt.show()

The middle of the plot sits on the red line — the main body of residuals is essentially normal. But the **upper right** curves above the line: the right tail is heavier than a true normal distribution.

Is this a disaster? No. The deviation is mild, concentrated in the tails, and we have 392 observations. The Central Limit Theorem handles this. We'd flag it but wouldn't throw out the model.

The two extreme failures to watch for:

- **Both tails curving sharply away** → very heavy tails, possibly outliers (coming in 17_1_4).
- **S-shape** → badly skewed residuals, often fixable with a transformation (17_1_5).

---

## Section 4 — E: Equal Variance (Homoscedasticity)

> **The assumption:** the spread of residuals is constant across all fitted values.

The opposite — **heteroscedasticity** — means the formula for the standard error of the slope is wrong, so p-values and CIs can be miscalibrated.

### What a pure failure looks like

A **funnel shape** (or "megaphone") in the residual plot. Let's build a synthetic example so the pattern is unmistakable.

In [ ]:
# Synthetic data: noise std grows with x.
x_fake = np.linspace(0, 20, 400)
noise_scale = 0.3 + 0.6 * x_fake
y_fake = 2 + 1.5 * x_fake + rng.normal(0, noise_scale, size=len(x_fake))
model_fake = sm.OLS(y_fake, sm.add_constant(x_fake)).fit()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].scatter(x_fake, y_fake, s=12, alpha=0.5, color='steelblue')
axes[0].plot(x_fake, model_fake.predict(sm.add_constant(x_fake)),
             color='darkorange', linewidth=2.5, label='OLS fit')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Fake data: noise grows with x')
axes[0].legend()

axes[1].scatter(model_fake.fittedvalues, model_fake.resid, s=12, alpha=0.5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Fitted value $\\hat{y}$')
axes[1].set_ylabel('Residual')
axes[1].set_title('Textbook funnel = heteroscedasticity')

plt.tight_layout()
plt.show()

The residuals form a clean funnel: narrow on the left, wide on the right. This is **heteroscedasticity** in its purest form, with no non-linearity to distract. Once you've seen this shape, you'll spot it everywhere.

What would a fix look like? If we apply a log transformation to $y$, the multiplicative compression often stabilizes variance:

In [ ]:
# Demonstrate a fix: log(y) compresses the scale, reducing the funnel.
# (For this dataset the fix is partial; in 17_1_5 we'll see cleaner fixes.)
model_log_fake = sm.OLS(np.log(np.abs(y_fake) + 1), sm.add_constant(x_fake)).fit()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

axes[0].scatter(model_fake.fittedvalues, model_fake.resid, s=12, alpha=0.5, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Fitted $\\hat{y}$')
axes[0].set_ylabel('Residual')
axes[0].set_title('Raw model: funnel present')

axes[1].scatter(model_log_fake.fittedvalues, model_log_fake.resid, s=12, alpha=0.5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Fitted $\\hat{y}$')
axes[1].set_ylabel('Residual')
axes[1].set_title('After log(y): funnel reduced')

plt.tight_layout()
plt.show()

The funnel is noticeably reduced after the log transform. In Notebook 17_1_5 we'll explore transformations more systematically and see how they can fix both non-linearity and heteroscedasticity at the same time.

### And in our real MPG data?

In [ ]:
# Compare residual spread at low vs. high fitted values.
resid = model.resid
fitted = model.fittedvalues
left = resid[fitted < fitted.median()]
right = resid[fitted >= fitted.median()]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(fitted, resid, s=20, alpha=0.45, color='steelblue')
ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Fitted mpg  $\\hat{y}$')
ax.set_ylabel('Residual (mpg)')
ax.set_title('MPG residuals vs. fitted')
plt.show()

print(f'Std of residuals (low fitted values): {left.std():.2f}')
print(f'Std of residuals (high fitted values): {right.std():.2f}')

Two things are happening simultaneously:

1. **The U-shape from Section 1 is here.** That's the **L** failure.
2. **The spread is not constant.** The vertical span is noticeably wider on the right (high mpg predictions, small-engine cars) than on the left. The standard deviation of residuals roughly doubles (from about 3 to about 6 mpg). That's the **E** failure.

If you're having trouble seeing the funnel under the U-shape, look at the *vertical scatter* at a fitted value of 15 vs. 35 — the band is roughly twice as wide at the right end.

The MPG model is failing **L** and **E**. Both failures bias the standard errors and p-values that statsmodels reported.

The transformations that can fix these violations are covered in **17_1_5** (log, reciprocal, Box-Cox). The influence diagnostics that identify whether specific rows are causing the failures are covered in **17_1_4**.

---

## Putting It All Together

Every LINE assumption follows the same diagnostic recipe:

> **Fit the model → extract residuals → look at them. If they form a pattern, an assumption is broken.**

| Letter | Assumption | How to check | Failure looks like |
|:-:|---|---|---|
| **L** | Linearity | `sns.residplot(x, y, lowess=True)` | U-curve or bend |
| **I** | Independence | Durbin-Watson (ordered data); plot vs. time/group | DW far from 2; correlated neighbors |
| **N** | Normality | `sm.qqplot(resid, line='s')` | Points curve off the reference line |
| **E** | Equal variance | Residuals vs. fitted scatter | Funnel / megaphone shape |

And the one-sentence takeaway:

> **A regression summary table is a mathematical result, not a promise. The slope, p-value, and confidence interval are only as trustworthy as the assumptions they're built on. Always check the residuals.**

### What to do when assumptions fail

| Failure | Common fixes |
|---|---|
| **Linearity** (U or curve) | Transform $x$ and/or $y$ (log, reciprocal, square root). Add polynomial terms. Use a nonlinear model. |
| **Independence** (serial correlation) | Time-series model (ARIMA); mixed-effects / clustered errors. |
| **Normality** (heavy tails, skew) | Often ignorable with large $n$ (CLT). Otherwise: transform $y$, or use robust methods. |
| **Equal variance** (funnel) | Transform $y$ (log often helps). Or use robust standard errors (`get_robustcov_results()`). |

### Where We're Going Next

So far we've looked at the *overall shape* of residuals. But what about a single weird point? A lone outlier can drag the regression line, crater $R^2$, and change p-values — while the summary table stays silent. That's what `17_1_4_SLR_Influence.ipynb` is about: **leverage, outliers, and Cook's Distance**.

---

**YOUR TURN (10 min):** Load the Palmer Penguins data from 17_1_1 (`sns.load_dataset('penguins')`). Fit `body_mass_g ~ flipper_length_mm`. Generate the L, N, and E diagnostic plots. Which assumptions appear to hold? Which appear to be violated? Compare your answer to the discussion in 17_1_1.